# Prithvi WxC Downscaling with CORDEX Data: Model Inference

This notebook walks through running inference with a fine-tuned Prithvi downscaling model on CORDEX NZ data.

To replicate the results show in this notebook please download the required files from CORDEX-ML Bench (https://zenodo.org/records/17517423) repository

You need `git lfs` installed to download large files

---

## Setup

Python >= 3.10 is required

Make sure that your current working directory is `granite-wxc/`

In [1]:
import os
from pathlib import Path
REPO_ROOT = Path("/mnt/data2/kyo/granite-wxc").resolve()
os.chdir("/mnt/data2/kyo/granite-wxc/examples/CORDEX_ML")
print(f"Working directory set to: {Path.cwd()}")

Working directory set to: /mnt/data2/kyo/granite-wxc/examples/CORDEX_ML


In [ ]:
# ===================== USER PARAMETERS (EDIT ME) =====================
from pathlib import Path
from nz_params import UserParams, validate_paths

REPO_ROOT = Path("/mnt/data2/kyo/granite-wxc").resolve()
PROJECT_DIR = REPO_ROOT / "examples/CORDEX_ML"
DATASET_ROOT = REPO_ROOT / "granite-geospatial-wxc-downscaling/CORDEX/NZ_domain"

TRAIN_SPLIT = "train/ESD_pseudo_reality"
TEST_SPLIT = "test/historical/predictors/perfect"

# Optional: use a single predictor file. Set to None to use all predictors in TEST_SPLIT.
PREDICTOR_FILE = "ACCESS-CM2_1981-2000_regridded.nc"

# Template target file used for grid/metadata and time alignment (not the output name).
TARGET_TEMPLATE_FILE = "pr_tasmax_ACCESS-CM2_1961-1980.nc"

TRAIN_TARGETS = [DATASET_ROOT / TRAIN_SPLIT / "target" / TARGET_TEMPLATE_FILE]

if PREDICTOR_FILE:
    TEST_PREDICTORS = [DATASET_ROOT / TEST_SPLIT / PREDICTOR_FILE]
else:
    TEST_PREDICTORS = sorted((DATASET_ROOT / TEST_SPLIT).glob('*.nc'))

FINETUNE_RUN_NAME = "NZ_T1_ACCESS-CM2_static"  # set None to auto-pick latest
USE_STATIC = True  # override config.data.use_static if not None

# Optional: override static path from config snapshot.
STATIC_PATH = None  # e.g., DATASET_ROOT / TRAIN_SPLIT / "predictors" / "Static_fields.nc"

REPAIR_INVALID_INPUTS = True
CLAMP_PR_NONNEGATIVE = True

# Optional: override prediction output filename
PREDICTION_OUTPUT_NAME = "Predictions_pr_tasmax_ACCESS-CM2_1981-2000.nc"

P = UserParams(
    repo_root=REPO_ROOT,
    project_dir=PROJECT_DIR,
    runs_root=PROJECT_DIR / "runs/NZ_T1_ACCESS-CM2_static_train",
    config_path=PROJECT_DIR / "NZ_T1_ACCESS-CM2_static.yaml",
    inference_run_name=FINETUNE_RUN_NAME,
    inference_output_root=PROJECT_DIR / "runs/NZ_T1_ACCESS-CM2_static_train/predictions/historical/perfect/",
    inference_predictor_root=DATASET_ROOT / TEST_SPLIT ,
    # NOTE: The notebook explicitly overrides config.data.test_* paths from the YAML.
    # Update these for each inference dataset; they take precedence over YAML defaults.
    test_predictor_paths=TEST_PREDICTORS,
    test_target_paths=TRAIN_TARGETS,
    use_static=USE_STATIC,
    checkpoint_path=None,
    preferred_checkpoint="best",
    device_target="cuda",
    num_workers=2,
)

validate_paths(P, require_inference=True)
print(P.summary())


FileNotFoundError: inference_predictor_root does not exist: /mnt/data2/kyo/granite-wxc/granite-geospatial-wxc-downscaling/CORDEX/NZ_domain/test/historical/predictors/perfect/predictors

In [ ]:
from pathlib import Path
import os

os.chdir(P.repo_root)
print(f"Working directory set to: {Path.cwd()}")

In [ ]:
import os

os.chdir(P.project_dir)
print(f"Project directory set to: {os.getcwd()}")

If your current directory is not `granite-wxc/`, change it using the following command:

```bash
%cd <local>/granite-wxc/
```

Replace `<local>` with the appropriate path prefix 

In [ ]:
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-WxC.git

In [ ]:
!pip install -q h5netcdf matplotlib wget pyyaml xarray scipy torch tqdm pysteps cartopy

In [ ]:
import logging
import warnings
logging.disable(logging.CRITICAL)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from contextlib import nullcontext
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

from cordex_inference import CordexWrappedDataset, build_inference_dataset, build_predictor_names
from utils.nearest_fill import repair_invalid_by_nearest_xr, summarize_invalid_counts_xr
from utils.postprocess_outputs import enforce_pr_nonnegative_xr
from granitewxc.utils.config import get_config
from granitewxc.models.model import get_finetune_model_UNET, get_finetune_model


Configure the backends, PyTorch states, and random seeds to standardize the RNG for random crops in this example

In [ ]:
torch.jit.enable_onednn_fusion(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed(42)
torch.manual_seed(42)
np.random.seed(42)

It is possible to use a cpu or gpu/s to generate inferences. Based on avaiablity of a `cuda:gpu`, we set the device that the model uses

In [ ]:
target = (getattr(P, 'device_target', 'auto') or 'auto').lower()
if target == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError("device_target is 'cuda' but no CUDA device is available.")
    device = torch.device('cuda')
elif target == 'cpu':
    device = torch.device('cpu')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
import os
from pathlib import Path

from run_utils import (
    assert_no_eccc_reference,
    load_run_manifest,
)
from nz_params import resolve_existing_run_dir, resolve_checkpoint, export_params

RUN_NAME, run_dir = resolve_existing_run_dir(P)
manifest = load_run_manifest(run_dir)
resolved_config_path = Path(manifest['config_snapshot']).resolve()
os.chdir(P.project_dir)
config = get_config(str(resolved_config_path))
assert_no_eccc_reference(resolved_config_path)

# Ensure scalers point at the run's scalar directory (manifest records the canonical paths).
manifest_scalars = manifest.get("scalars", {})
if isinstance(manifest_scalars, dict):
    model_scaler_map = {
        "model.input_mu": "input_mu",
        "model.input_sigma": "input_sigma",
        "model.target_mu": "target_mu",
        "model.target_sigma": "target_sigma",
    }
    for key, attr in model_scaler_map.items():
        path = manifest_scalars.get(key)
        if path:
            setattr(config.model, attr, path)
    data_scalers = manifest_scalars.get("data.scalers")
    if data_scalers:
        config.data.scalers = data_scalers

if P.num_workers is not None:
    config.dl_num_workers = int(P.num_workers)
if P.batch_size is not None:
    config.batch_size = int(P.batch_size)
config.device_target = getattr(P, 'device_target', config.device_target)
if getattr(P, "use_static", None) is not None:
    config.data.use_static = bool(P.use_static)
if STATIC_PATH:
    config.data.static_path = str(Path(STATIC_PATH).resolve())

checkpoint_path = resolve_checkpoint(P, run_dir)
assert_no_eccc_reference(checkpoint_path)

output_root = Path(P.inference_output_root or (run_dir / 'predictions'))
OUTPUT_DIR = output_root
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_output_stub = Path(PREDICTION_OUTPUT_NAME or f"{RUN_NAME}_predictions.nc")


params_json = export_params(
    P,
    OUTPUT_DIR / prediction_output_stub.with_suffix(".json"),
    extra={'run_name': RUN_NAME, 'checkpoint': str(checkpoint_path)},
)

print(f"Using fine-tune run '{RUN_NAME}' located at {run_dir}")
print(f"Resolved checkpoint: {checkpoint_path}")
print(f"Inference artifacts will be written to {OUTPUT_DIR}")
print(f"Saved parameter snapshot to {params_json}")

CHECKPOINT_PATH = checkpoint_path


## Configuration File

The model is configured using `YAML` files.

In these files, you specify:
- Paths to the input data  
- Locations of the pretrained weights  

To ensure compatibility with the provided weights during inference, keep the model configuration consistent with the original definitions 

In [ ]:
from pathlib import Path

def _resolve_inputs(values):
    return [str(Path(p).resolve()) for p in values]

test_predictor_paths = _resolve_inputs(P.test_predictor_paths) if P.test_predictor_paths else []
if not test_predictor_paths and P.inference_predictor_root:
    test_predictor_paths = [str(path.resolve()) for path in Path(P.inference_predictor_root).glob('*.nc')]

if not test_predictor_paths:
    raise FileNotFoundError('No inference predictor files configured in USER PARAMETERS.')

config.data.test_predictor_paths = test_predictor_paths
if P.test_target_paths:
    config.data.test_target_paths = _resolve_inputs(P.test_target_paths)

root_dir = Path(test_predictor_paths[0]).parent
print(f"Using perfect predictors from {root_dir} ({len(test_predictor_paths)} files)")

In [ ]:
preproc_cache = Path(manifest["preproc_dir"]).resolve()
print(f"Preprocessed predictor cache (shared with fine-tune run): {preproc_cache}")


## Dataloader 

We reuse the same regridded CORDEX sample for testing to showcase the end-to-end pipeline.

In [ ]:

def _repair_predictor_files_if_needed(predictor_paths, predictor_var_names):
    global REPAIR_DIAGNOSTICS
    resolved_paths = [str(Path(path).resolve()) for path in predictor_paths]
    REPAIR_DIAGNOSTICS = []

    if not REPAIR_INVALID_INPUTS:
        print("[repair] Predictor invalid-value repair disabled by REPAIR_INVALID_INPUTS=False")
        for path in resolved_paths:
            REPAIR_DIAGNOSTICS.append({"file": path, "before": {}, "after": {}})
        return resolved_paths

    repaired_dir = OUTPUT_DIR / "_repaired_predictors"
    repaired_dir.mkdir(parents=True, exist_ok=True)

    final_paths = []
    for predictor_path in resolved_paths:
        source_path = Path(predictor_path)
        with xr.open_dataset(source_path, engine="netcdf4") as ds:
            predictor_ds = ds.load()

        before_counts = summarize_invalid_counts_xr(predictor_ds, var_names=predictor_var_names)
        invalid_before_total = sum(before_counts.values())

        if invalid_before_total > 0:
            print("Invalid predictor values detected (NaN/inf/fill). Auto-repair enabled.")
            predictor_ds.encoding["source"] = str(source_path)
            repaired_ds = repair_invalid_by_nearest_xr(
                predictor_ds,
                var_names=predictor_var_names,
            )
            after_counts = summarize_invalid_counts_xr(repaired_ds, var_names=predictor_var_names)

            repaired_path = repaired_dir / source_path.name
            repaired_ds.to_netcdf(repaired_path, engine="h5netcdf")
            final_paths.append(str(repaired_path.resolve()))
        else:
            after_counts = before_counts
            final_paths.append(str(source_path))

        for key in sorted(set(before_counts) | set(after_counts)):
            print(
                f"[repair] {source_path.name} {key}: "
                f"invalid_count_before={before_counts.get(key, 0)} "
                f"invalid_count_after={after_counts.get(key, 0)}"
            )

        REPAIR_DIAGNOSTICS.append(
            {
                "file": str(source_path),
                "before": before_counts,
                "after": after_counts,
            }
        )

    return final_paths


def build_dataloader(predictor_paths, target_paths):
    predictor_var_names = build_predictor_names(config)
    safe_predictor_paths = _repair_predictor_files_if_needed(predictor_paths, predictor_var_names)
    base_dataset = build_inference_dataset(config, safe_predictor_paths, target_paths)
    dataset = CordexWrappedDataset(base_dataset)
    batch_size = getattr(config, 'batch_size', 1)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=config.dl_num_workers,
        pin_memory=(device.type == 'cuda'),
    )


In [ ]:
test_dl = build_dataloader(
    config.data.test_predictor_paths, config.data.test_target_paths
)

In [ ]:
base_dataset = test_dl.dataset.base
inference_target_vars = list(base_dataset.target_vars)
print(f"Inference target variables: {inference_target_vars}")
if set(inference_target_vars) != {"pr", "tasmax"}:
    raise ValueError(
        "Inference dataloader must expose both 'pr' and 'tasmax' targets; update the config/output_vars list."
    )
print(f"Total inference samples: {len(base_dataset)} (time steps across {len(base_dataset.predictor_paths)} predictor file(s))")


## Model Initialization

We provide **2** different model architectures `UNET-like` and `CONV` 

Both architectures include:  
1. **Patch Embedding**: Extracts shallow features from the input data  
2. **Feature Extraction**: Utilizes the Prithvi backbone to extract deeper features  

The key difference is that the UNET-like version incorporates **static high-resolution data** into the model

In this notebook, we use the **UNET-like** version

To switch to the **CONV** model, update the configuration file accordingly and use `get_finetune_model(config)`

In [ ]:
model = get_finetune_model_UNET(config)

In [ ]:
from pathlib import Path

assert_no_eccc_reference(CHECKPOINT_PATH)
checkpoint = torch.load(str(CHECKPOINT_PATH), map_location='cpu', weights_only=False)
state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
model_state = model.state_dict()
weights_have_module_prefix = all(key.startswith('module.') for key in state_dict.keys())
model_expects_module_prefix = all(key.startswith('module.') for key in model_state.keys())

if model_expects_module_prefix and not weights_have_module_prefix:
    state_dict = state_dict.__class__((('module.' + key), value) for key, value in state_dict.items())
elif weights_have_module_prefix and not model_expects_module_prefix:
    prefix_len = len('module.')
    state_dict = state_dict.__class__((key[prefix_len:], value) for key, value in state_dict.items())

model.load_state_dict(state_dict, strict=True)
model.to(device)

skip_offload_devices = []
if device.type == 'cuda' and torch.cuda.device_count() > 1:
    skip_offload_devices = [torch.device(f'cuda:{idx}') for idx in range(1, torch.cuda.device_count())]
    if skip_offload_devices and hasattr(model, 'set_skip_activation_devices'):
        model.set_skip_activation_devices(skip_offload_devices)
        print(f"--> Offloading skip activations to GPUs: {skip_offload_devices}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
first_tensor = next(iter(model.state_dict().values()))
mean_val = first_tensor.float().mean().item()
std_val = first_tensor.float().std().item()

print(f"Loaded fine-tuned weights from {CHECKPOINT_PATH}")
print(f"Trainable parameter count: {total_params:,}")
print(f"First state_dict tensor stats -> mean: {mean_val:.6f}, std: {std_val:.6f}")

We can now load the pretrained weights

In [ ]:
# Checkpoint loading handled earlier with CHECKPOINT_PATH.

### Inference

The model is now ready for inference. We run inference for one batch (batch_size=1).

In [ ]:
with torch.no_grad():
    model.eval()
    
    batch = next(iter(test_dl))
    batch = {k: v.to(device) for k, v in batch.items()}
    autocast_enabled = device.type == "cuda"
    autocast_dtype = torch.bfloat16 if (autocast_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    autocast_ctx = torch.cuda.amp.autocast(dtype=autocast_dtype, enabled=autocast_enabled) if autocast_enabled else nullcontext()
    
    with autocast_ctx:
        out = model(batch)

    inputs = batch['x']
    targets = batch['y']
    outputs = out

In [ ]:
inputs.shape, targets.shape, outputs.shape

inputs.shape, targets.shape, outputs.shape

In [ ]:
### Plotting

#We visualise one of the output channels to inspect the prediction quality.

In [ ]:
var_names = list(config.data.output_vars)
var_idx = 0  # pr
sample_idx = 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(targets[sample_idx, var_idx].cpu(), cmap='coolwarm')
axes[0].set_title(f'Target {var_names[var_idx]}')
axes[0].axis("off")
axes[1].imshow(outputs[sample_idx, var_idx].cpu(), cmap='coolwarm')
axes[1].set_title(f'Prediction {var_names[var_idx]}')
axes[1].axis("off")
plt.show()

In [ ]:
prediction_output_path = OUTPUT_DIR / prediction_output_stub
prediction_output_path.parent.mkdir(parents=True, exist_ok=True)

base_dataset = test_dl.dataset.base
time_dim = base_dataset.time_dim or 'time'
lat_dim, lon_dim = base_dataset.output_spatial_dims
lat_name = base_dataset.fine_lat_name
lon_name = base_dataset.fine_lon_name
target_vars = list(base_dataset.target_vars)
predictor_paths = list(base_dataset.predictor_paths)
target_template_paths = list(base_dataset.target_paths)

def _run_full_inference(dataloader, model, device):
    predictions = []
    autocast_enabled = device.type == 'cuda'
    autocast_dtype = (
        torch.bfloat16 if (autocast_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    )

    def autocast_context():
        return (
            torch.cuda.amp.autocast(dtype=autocast_dtype, enabled=autocast_enabled)
            if autocast_enabled
            else nullcontext()
        )

    with torch.no_grad():
        model.eval()
        for batch in tqdm(dataloader, desc='Running inference', leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast_context():
                out = model(batch)
            predictions.append(out.detach().cpu())
    return torch.cat(predictions, dim=0)


def _concat_time_coordinate(paths, time_key):
    arrays = []
    attrs = None
    encoding = None
    for path in paths:
        with xr.open_dataset(path, engine='netcdf4') as ds:
            if time_key not in ds.coords and time_key not in ds.data_vars:
                raise KeyError(f"Time dimension '{time_key}' missing in {path}")
            da = ds[time_key]
            arrays.append(da)
            if attrs is None:
                attrs = dict(da.attrs)
            if encoding is None and hasattr(da, 'encoding'):
                encoding = dict(da.encoding)
    combined = xr.concat(arrays, dim=time_key)
    if attrs:
        combined.attrs.update(attrs)
    if encoding:
        combined.encoding.update(encoding)
    return combined.load()


full_outputs = _run_full_inference(test_dl, model, device)
outputs_np = full_outputs.numpy()
if outputs_np.shape[1] != len(target_vars):
    raise ValueError(
        f"Model produced {outputs_np.shape[1]} channels but target_vars expects {len(target_vars)}"
    )

predictor_time_coord = _concat_time_coordinate(predictor_paths, time_dim)
if outputs_np.shape[0] != predictor_time_coord.sizes[time_dim]:
    raise ValueError(
        f"Prediction time dimension {outputs_np.shape[0]} does not match predictor timestamps {predictor_time_coord.sizes[time_dim]}"
    )

start_time = predictor_time_coord.values[0]
end_time = predictor_time_coord.values[-1]
print(
    f"Predictor-driven time axis spans {start_time} to {end_time} ({predictor_time_coord.sizes[time_dim]} steps)."
)

if not target_template_paths:
    raise FileNotFoundError('Target template paths missing; ensure config.data.test_target_paths is set')

with xr.open_dataset(target_template_paths[0], engine='h5netcdf') as template_ds:
    template_attrs = dict(template_ds.attrs)
    target_attrs = {
        name: dict(template_ds[name].attrs)
        for name in target_vars
        if name in template_ds.data_vars
    }
    lat_coord = template_ds[lat_name].load()
    lon_coord = template_ds[lon_name].load()


In [ ]:
coords = {
    time_dim: predictor_time_coord,
    lat_dim: lat_coord,
    lon_dim: lon_coord,
}

prediction_ds = xr.Dataset(coords=coords)
for idx, name in enumerate(target_vars):
    prediction_ds[name] = xr.DataArray(
        outputs_np[:, idx],
        dims=(time_dim, lat_dim, lon_dim),
        coords=coords,
        attrs=target_attrs.get(name, {}),
    )

CLAMP_DIAGNOSTICS = {}
if CLAMP_PR_NONNEGATIVE and "pr" in prediction_ds.data_vars:
    pr_before = prediction_ds["pr"]
    neg_before = int((pr_before < 0).sum().item())
    min_before = float(pr_before.min().item())
    prediction_ds = enforce_pr_nonnegative_xr(prediction_ds)
    pr_after = prediction_ds["pr"]
    neg_after = int((pr_after < 0).sum().item())
    min_after = float(pr_after.min().item())
    CLAMP_DIAGNOSTICS = {
        "negative_count_before": neg_before,
        "negative_count_after": neg_after,
        "min_before": min_before,
        "min_after": min_after,
    }
elif not CLAMP_PR_NONNEGATIVE:
    print("[clamp] Precipitation clamp disabled by CLAMP_PR_NONNEGATIVE=False")
else:
    print("[clamp] Could not find pr variable for clamping")

prediction_ds.attrs.update(template_attrs)
prediction_ds.to_netcdf(prediction_output_path, engine="h5netcdf")
print(
    f"Saved predictions to {prediction_output_path} with time range {predictor_time_coord.values[0]} to {predictor_time_coord.values[-1]}"
)


In [ ]:

print("[diagnostics] Predictor invalid counts before/after repair")
for record in REPAIR_DIAGNOSTICS:
    file_name = Path(record["file"]).name
    keys = sorted(set(record["before"]) | set(record["after"]))
    if not keys:
        print(f"  {file_name}: no tracked predictor variables")
        continue
    for key in keys:
        print(
            f"  {file_name} {key}: "
            f"invalid_count_before={record['before'].get(key, 0)} "
            f"invalid_count_after={record['after'].get(key, 0)}"
        )

if CLAMP_DIAGNOSTICS:
    print("[diagnostics] Precipitation clamp")
    print(
        "  pr: "
        f"negative_count_before={CLAMP_DIAGNOSTICS['negative_count_before']} "
        f"negative_count_after={CLAMP_DIAGNOSTICS['negative_count_after']} "
        f"min_before={CLAMP_DIAGNOSTICS['min_before']:.6g} "
        f"min_after={CLAMP_DIAGNOSTICS['min_after']:.6g}"
    )
else:
    print("[diagnostics] Precipitation clamp diagnostics unavailable")


In [ ]:
import pickle

pickle_path = OUTPUT_DIR / prediction_output_stub.with_suffix(".pkl")
with open(pickle_path, 'wb') as handle:
    pickle.dump(outputs_np, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved raw predictions array to {pickle_path}")